# 05_訓練自己的YOLO.py
**電腦視覺與深度學習 › Day20 YOLO11n 物件偵測(預訓練到自己訓練)**  ·  ⚠️ 錯誤示範(有 bug,對照修正版看)

這支程式訓練你自己的 YOLO11n 模型:自動找資料集、偵測有沒有 GPU、建議批次大小,再開始訓練。這是整個課程的高潮。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/7/4)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> ⚡ 這支會用到深度學習,建議先到「執行階段 › 變更執行階段類型」選 **T4 GPU**。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week2/DAY10"
!pip -q install ultralytics
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '05_訓練自己的YOLO.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 修正 data.yaml 路徑(檔案裡寫的是講師電腦的 C:/ 路徑)
import re, pathlib
for y in pathlib.Path('datasets').rglob('data.yaml'):
    t = y.read_text(encoding='utf-8')
    y.write_text(re.sub(r'(?m)^path:.*$', 'path: ' + y.parent.resolve().as_posix(), t), encoding='utf-8')
    print('已修正', y)

### 說明與時間參考
註解列出訓練時間參考:同樣 20 輪,筆電 CPU 要 30 到 90 分鐘,好一點的顯示卡只要幾分鐘。沒有 GPU 也能跑,只是要有耐心。

In [ ]:
# 05 — 訓練你自己的 YOLO11n

### 匯入與路徑
匯入 torch 和 YOLO。資料集放在 datasets 資料夾,訓練結果固定存到這個資料夾底下的 runs,不管你從哪裡執行。

In [ ]:
#
# 這支腳本會：
#   1) 自動掃 datasets/ 底下所有 data.yaml，列給你選
#   2) 偵測有沒有 GPU，沒有就 CPU 訓練（會慢很多）
#   3) 依 GPU 顯存自動建議 batch size
#   4) 訓練結果永遠存到 DAY10/runs/ 底下（絕對路徑，跟你 cwd 無關）
#
# 訓練時間參考（YOLO11n、640x640、20 epochs、1200 張左右資料集）：
#   - CPU (i5/i7 筆電)          30-90 分鐘
#   - GTX 1660 6GB              6-10 分鐘
#   - RTX 3060 12GB             3-5 分鐘
#   - RTX 4090 24GB             1-2 分鐘
#
# 環境確認：
#   pip install -r requirements.txt

from pathlib import Path
import sys

import torch
from ultralytics import YOLO

BASE       = Path(__file__).parent
DATASETS   = BASE / "datasets"

### 訓練參數
EPOCHS 是訓練幾輪,IMAGE_SIZE 是圖片大小,PATIENCE 是連續幾輪沒進步就提早停止,避免過擬合。

In [ ]:
PROJECT_DIR = BASE / "runs"        # 絕對路徑，訓練結果永遠在 DAY10/runs/


# ====== 訓練參數（可微調）======
EPOCHS       = 20         # 訓練回合。20 是入門合理值。看 mAP 曲線沒收斂再加
IMAGE_SIZE   = 640        # 影像解析度。320 快、640 準、1280 GPU 才吃得動
RUN_NAME     = "train"    # 存到 runs/train, runs/train2, ...

### 掃描所有資料集
用 rglob 在 datasets 底下找所有 data.yaml,每找到一個就是一個資料集,用它的資料夾名稱當名字。

In [ ]:
PATIENCE     = 10         # 連續 N epoch mAP 沒進步就早停（early stopping）


# ====== 自動找可用的 dataset ======
def 找所有dataset():
    """回傳 [(name, data_yaml_path), ...]"""
    if not DATASETS.exists():
        return []
    results = []
    for yaml in sorted(DATASETS.rglob("data.yaml")):
        # yaml 檔案的父資料夾名字當 dataset 名字
        results.append((yaml.parent.name, yaml))
    return results

### 選一個資料集
一個都沒有,就教你兩種取得方法並結束。只有一個就直接用。有很多個就列出來,顯示類別數和名稱,讓你輸入編號,直接按 Enter 用第一個。

### ⚠️ 依顯示卡記憶體建議批次大小
沒有 GPU 就用 CPU、批次 8。有 GPU 就看顯示記憶體大小決定批次。小提醒:註解說每個批次大約要 1.5 GB,照這樣算,6 GB 的卡用 8 會不夠,註解和實際的門檻對不太起來。如果訓練中出現記憶體不足,就把批次調小。

In [ ]:
def 選dataset(datasets):
    if not datasets:
        print()
        print("=" * 60)
        print("找不到任何 dataset")
        print("=" * 60)
        print(f"預期位置: {DATASETS}/<名字>/data.yaml")
        print()
        print("有兩條路取得資料集：")
        print("  A) 跑 04_下載範例dataset.py 從 Roboflow 下載")
        print("  B) 用 Label Studio 標完後，跑 convert_ls_json_to_yolo.py 轉檔")
        sys.exit(1)

    if len(datasets) == 1:
        name, path = datasets[0]
        print(f"只找到一個 dataset：{name}")
        return path

    print()
    print("找到多個 dataset，選一個訓練：")
    for i, (name, path) in enumerate(datasets, 1):
        # 順便顯示類別資訊
        try:
            import yaml as yaml_lib
            info = yaml_lib.safe_load(open(path, encoding="utf-8"))
            nc = info.get("nc", "?")
            names = info.get("names", [])
            print(f"  [{i}] {name:20s}  {nc} 類：{names}")
        except Exception:
            print(f"  [{i}] {name}")
    print()

    while True:
        try:
            choice = input(f"選 [1-{len(datasets)}]（Enter 用 1）: ").strip()
        except (EOFError, KeyboardInterrupt):
            sys.exit(1)
        if not choice:
            return datasets[0][1]
        try:
            idx = int(choice) - 1
            if 0 <= idx < len(datasets):
                return datasets[idx][1]
        except ValueError:
            pass
        print("請輸入有效數字")


# ====== 依 GPU 顯存建議 batch size ======
def 建議batch_size():
    if not torch.cuda.is_available():
        return 8, "cpu", None
    device_name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

    # 粗略對照：640 訓練 YOLO11n 大約需要 (batch × 1.5) GB
    if total_gb >= 20:
        batch = 32
    elif total_gb >= 10:
        batch = 16
    elif total_gb >= 6:
        batch = 8
    elif total_gb >= 4:
        batch = 4
    else:
        batch = 2
    return batch, "cuda:0", f"{device_name} ({total_gb:.1f} GB)"

### 選資料集、顯示裝置
主程式先選資料集,再顯示偵測到的 GPU 和建議的批次大小,以及這次的訓練參數。

### 按 Enter 才開始
訓練可能很久,所以先停下來等你按 Enter,避免不小心啟動。

### 開始訓練
從預訓練的 yolo11n 出發,呼叫 train。這就是遷移學習:不是從零開始,而是在已經會看東西的模型上,學你的新類別。exist_ok 等於 False,每次訓練都會開一個新資料夾。

### 結果在哪裡
訓練完印出最佳權重 best.pt、最後權重 last.pt、訓練圖表和混淆矩陣的位置,並提醒下一步用 06 評估、07 或 08 推論。

In [ ]:
def main():
    print("=" * 60)
    print("YOLO11n 訓練 (Ultralytics)")
    print("=" * 60)

    # 1) 選 dataset
    datasets = 找所有dataset()
    data_yaml = 選dataset(datasets)
    print(f"使用 dataset：{data_yaml}")

    # 2) 顯示裝置與 batch size 建議
    batch, device, gpu_info = 建議batch_size()
    print()
    if gpu_info:
        print(f"偵測到 GPU：{gpu_info}")
        print(f"建議 batch size：{batch}")
        print("  若訓練中 out of memory，把 batch 調小（或關其他吃 GPU 的程式）")
    else:
        print("沒偵測到 GPU，使用 CPU 訓練")
        print(f"建議 batch size：{batch}（CPU 也不宜太大）")
        print("  預估時間：小資料集 30-60 分鐘，中資料集 1-3 小時")

    print()
    print(f"訓練參數：epochs={EPOCHS}, imgsz={IMAGE_SIZE}, patience={PATIENCE}")
    print(f"結果存到：{PROJECT_DIR}")
    print()

    # 3) 讓學員確認（避免不小心啟動長時間訓練）
    try:
        proceed = input("按 Enter 開始訓練（Ctrl+C 取消）：").strip()
    except (EOFError, KeyboardInterrupt):
        sys.exit(0)

    # 4) 訓練
    model = YOLO("yolo11n.pt")   # 第一次跑會自動下載 5.5MB
    results = model.train(
        data       = str(data_yaml),
        epochs     = EPOCHS,
        imgsz      = IMAGE_SIZE,
        batch      = batch,
        device     = device,
        project    = str(PROJECT_DIR),   # 絕對路徑！不管 cwd 在哪都存這
        name       = RUN_NAME,
        exist_ok   = False,              # 每次跑新 folder：train, train2, ...
        patience   = PATIENCE,
        verbose    = True,
    )

    # 5) 結果摘要
    save_dir = Path(results.save_dir)
    print()
    print("=" * 60)
    print("訓練完成")
    print("=" * 60)
    print(f"最佳權重：{save_dir / 'weights' / 'best.pt'}")
    print(f"最後權重：{save_dir / 'weights' / 'last.pt'}")
    print(f"訓練圖表：{save_dir / 'results.png'}")
    print(f"混淆矩陣：{save_dir / 'confusion_matrix.png'}")
    print()
    print("下一步：")
    print("  06_評估訓練結果.py     看 mAP / Loss 曲線")
    print("  07_自訓練模型推論.py   接 webcam 有 GUI 版")
    print("  08_webcam串流推論.py   接 webcam 串流到瀏覽器（推薦）")


if __name__ == "__main__":
    main()

## 小結
你訓練出了自己的模型。下一支 06 會讀訓練紀錄,畫出曲線,看它訓練得好不好。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/7/4) 看逐句拆解、做練習。